# Imports

In [ ]:
import numpy as np
import pandas as pd
import pulp
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import truncnorm


# Loading data


In [ ]:
data_dir = r"..\data\ToyCar_StudentProject.xlsx"

In [ ]:
equipements = pd.read_excel(data_dir, "Equipements", index_col=0)
generic = pd.read_excel(data_dir, "Generic", index_col=0)
operations = pd.read_excel(data_dir, "Operations", index_col="Operation")

In [ ]:
# equipements
equipement_id = equipements.index.to_list()
category = equipements['Category']
supplier = equipements['Supplier']
if 'Age' in equipements.columns:
    age = equipements['Age']
speed_factor = equipements['Speed factor']
purshasing_cost = equipements['Purchasing cost']
operation_cost = equipements['Operation cost']
salvage_cost = equipements['Salvage cost ']
# generic
takt = generic[generic.index=="Takt"]['Value'].values
horizon = generic[generic.index=="Horizon"]['Value'].values
lost_sale_penalty = generic[generic.index=="LostSalePenalty"]['Value'].values
years = list(range(1, int(horizon[0]) + 1))
duration_year_seconds = 24 * 3600 * 365
# operations
resource_group = operations['Resource Group'].unique()
operation = operations.index.unique().to_list()
mean_duration_operation_year = operations.iloc[:, 1:].to_numpy(dtype=float)
if mean_duration_operation_year.shape[1] != len(years):
    raise ValueError("the number of duration columns must match the planning horizon")

# Scenario generation

In [ ]:
N_scenarios = 10

In [ ]:
mean_duration_operation_year

In [ ]:
def generate_scenarios(t_base, n, distribution="uniform", sigma_ratio=0.2):
    """generate an array of shape (n, n_operations, n_years)."""
    t_base = np.asarray(t_base, dtype=float)
    if n < 1 or np.any(t_base < 0):
        raise ValueError("n must be positive and operation durations nonnegative")

    shape = (n, *t_base.shape)
    if distribution == "uniform":
        return np.random.uniform(t_base, 2 * t_base, size=shape)

    if distribution == "truncated_normal":
        if sigma_ratio <= 0:
            raise ValueError("sigma_ratio must be positive")
        result = np.zeros(shape, dtype=float)
        positive = t_base > 0
        if np.any(positive):
            means = t_base[positive]
            std = means * sigma_ratio
            result[:, positive] = truncnorm.rvs(
                a=np.zeros_like(means),
                b=means / std,
                loc=means,
                scale=std,
                size=(n, len(means)),
            )
        return result

    raise ValueError("distribution must be 'uniform' or 'truncated_normal'")


In [ ]:
N_scenarios = 10
scenarios = range(N_scenarios)
np.random.seed(42)
duration_operation_year_scenarios = generate_scenarios(
    mean_duration_operation_year, N_scenarios, distribution="uniform"
)


# Modeling

In [ ]:
def solve_model(durations, solver=None, fixed_y=None):
    """solve the two-stage stochastic model for the supplied scenarios."""
    durations = np.asarray(durations, dtype=float)
    if durations.ndim != 3 or durations.shape[1:] != mean_duration_operation_year.shape:
        raise ValueError("durations must have shape (n, n_operations, n_years)")
    n = len(durations)
    if n == 0:
        raise ValueError("at least one scenario is required")
    scenario_ids = range(n)

    model = pulp.LpProblem("Assembly_Line", pulp.LpMinimize)

    y_var = pulp.LpVariable.dicts(
        "Y", equipement_id, lowBound=0, cat="Integer"
    )
    if fixed_y is not None:
        for e in equipement_id:
            if e not in fixed_y:
                raise ValueError(f"missing fixed equipment quantity: {e}")
            value = float(fixed_y[e])
            if value < 0 or not np.isfinite(value) or not np.isclose(value, round(value)):
                raise ValueError(f"invalid equipment quantity for {e}: {value}")
            model += y_var[e] == round(value)

    x_var = pulp.LpVariable.dicts(
        "X", (equipement_id, resource_group, years, scenario_ids),
        lowBound=0, cat="Integer"
    )
    f_var = pulp.LpVariable.dicts(
        "F", (resource_group, years, scenario_ids),
        lowBound=0
    )
    m_var = pulp.LpVariable.dicts(
        "M", (years, scenario_ids), lowBound=0
    )

    capex = pulp.lpSum(
        y_var[e] * (purshasing_cost[e] - salvage_cost[e])
        for e in equipement_id
    )

    operating = pulp.lpSum(
        x_var[e][g][yr][s] * operation_cost[e] / n
        for e in equipement_id
        for g in resource_group
        for yr in years
        for s in scenario_ids
    )

    lost_sales = pulp.lpSum(
        m_var[yr][s] * duration_year_seconds
        * lost_sale_penalty[0] / n
        for yr in years
        for s in scenario_ids
    )

    model += capex + operating + lost_sales

    group_indices = {
        g: np.where(operations["Resource Group"].values == g)[0]
        for g in resource_group
    }

    for g in resource_group:
        indices = group_indices[g]

        for j, yr in enumerate(years):
            for s in scenario_ids:
                total = durations[s, indices, j].sum()

                if total > 0:
                    model += (
                        f_var[g][yr][s]
                        == pulp.lpSum(
                            x_var[e][g][yr][s] * speed_factor[e]
                            for e in equipement_id
                        ) / total
                    )
                else:
                    model += f_var[g][yr][s] == 0

    for e in equipement_id:
        for yr in years:
            for s in scenario_ids:
                model += (
                    pulp.lpSum(
                        x_var[e][g][yr][s]
                        for g in resource_group
                    ) <= y_var[e]
                )

    for g in resource_group:
        for yr in years:
            for s in scenario_ids:
                model += (
                    m_var[yr][s] >= 1 / takt[0] - f_var[g][yr][s]
                )

    if solver is None:
        solver = pulp.PULP_CBC_CMD(msg=False)

    model.solve(solver)

    if model.status != pulp.LpStatusOptimal:
        raise RuntimeError(
            f"Optimization failed: {pulp.LpStatus[model.status]}"
        )

    return {
        "model": model,
        "Y": y_var,
        "X": x_var,
        "F": f_var,
        "M": m_var,
        "scenarios": scenario_ids,
        "objective": pulp.value(model.objective)
    }



initial_solution = solve_model(duration_operation_year_scenarios)
problem = initial_solution["model"]
Y, X, F, M = (initial_solution[k] for k in ("Y", "X", "F", "M"))


# Solving the problem


In [ ]:
print(f"problem status: {pulp.LpStatus[problem.status]}")
print(f"objective function: {initial_solution['objective']:.4f}")
for var in problem.variables():
    if var.varValue is not None and abs(var.varValue) > 1e-9:
        print(f"{var.name}: {var.varValue}")


# Analysis

In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
def extract_frequency_data(solution):
    """extract optimized production frequencies."""
    data = []

    for g in resource_group:
        for yr in years:
            for s in solution["scenarios"]:
                data.append({
                    "group": g,
                    "year": yr,
                    "scenario": s,
                    "frequency": pulp.value(
                        solution["F"][g][yr][s]
                    )
                })

    return pd.DataFrame(data)


def extract_cost_data(solutions):
    """extract first-stage and expected second-stage costs."""
    data = []

    for approach, sol in solutions.items():
        y_var = sol["Y"]
        x_var = sol["X"]
        m_var = sol["M"]
        scenario_ids = sol["scenarios"]
        n = len(scenario_ids)

        first_stage = sum(
            pulp.value(y_var[e])
            * (purshasing_cost[e] - salvage_cost[e])
            for e in equipement_id
        )

        operating = sum(
            pulp.value(x_var[e][g][yr][s]) * operation_cost[e]
            for e in equipement_id
            for g in resource_group
            for yr in years
            for s in scenario_ids
        ) / n

        lost_sales = sum(
            pulp.value(m_var[yr][s])
            * duration_year_seconds * lost_sale_penalty[0]
            for yr in years
            for s in scenario_ids
        ) / n

        second_stage = operating + lost_sales

        data.extend([
            {
                "approach": approach,
                "cost_type": "1st stage (capex)",
                "value": first_stage
            },
            {
                "approach": approach,
                "cost_type": "expected 2nd stage",
                "value": second_stage
            }
        ])
        if not np.isclose(
            first_stage + second_stage,
            sol["objective"],
            rtol=1e-5,
            atol=1e-4
        ):
            raise ValueError(f"cost decomposition mismatch: {approach}")

    return pd.DataFrame(data)


def extract_convergence_data(
    ns=(5, 10, 15, 20),
    n_replications=1,
    seed=42
):
    """solve independent in-sample saa replications."""
    data = []
    rng = np.random.default_rng(seed)

    for distribution in ["uniform", "truncated_normal"]:
        for n in ns:
            for rep in range(n_replications):
                np.random.seed(
                    int(rng.integers(0, 2**31 - 1))
                )

                durations = generate_scenarios(
                    mean_duration_operation_year,
                    n,
                    distribution=distribution
                )

                solution = solve_model(durations)

                data.append({
                    "n_scenarios": n,
                    "distribution": (
                        "uniform" if distribution == "uniform"
                        else "truncated normal"
                    ),
                    "replication": rep,
                    "objective_value": solution["objective"]
                })

                print(
                    f"{distribution} | n={n} | "
                    f"rep={rep+1} | "
                    f"objective={solution['objective']:.2f}"
                )

    return pd.DataFrame(data)

def expected_durations_by_distribution(t_base, distribution, sigma_ratio=0.2):
    t_base = np.asarray(t_base, dtype=float)
    if distribution == "uniform":
        return 1.5 * t_base
    if distribution != "truncated_normal" or sigma_ratio <= 0:
        raise ValueError("invalid distribution or sigma_ratio")
    result = np.zeros_like(t_base)
    positive = t_base > 0
    if positive.any():
        means = t_base[positive]
        sigma = sigma_ratio * means
        result[positive] = truncnorm.mean(
            a=np.zeros_like(means), b=means / sigma,
            loc=means, scale=sigma
        )
    return result


def evaluate_stochastic_value(samples, distribution, sigma_ratio=0.2, rp_solution=None):
    rp_solution = rp_solution if rp_solution is not None else solve_model(samples)
    expected = expected_durations_by_distribution(
        mean_duration_operation_year, distribution, sigma_ratio
    )
    deterministic_solution = solve_model(expected[np.newaxis, :, :])
    fixed_y = {e: round(pulp.value(deterministic_solution["Y"][e])) for e in equipement_id}
    eev_solution = solve_model(samples, fixed_y=fixed_y)
    rp = float(rp_solution["objective"])
    eev = float(eev_solution["objective"])
    vss = eev - rp
    tolerance = 1e-5 * max(1, abs(rp), abs(eev))
    if vss < -tolerance:
        raise RuntimeError("eev must not be smaller than rp")
    return {
        "RP": rp,
        "deterministic_cost": float(deterministic_solution["objective"]),
        "EEV": eev,
        "VSS": max(0.0, vss),
        "rp_solution": rp_solution,
        "deterministic_solution": deterministic_solution,
        "eev_solution": eev_solution,
        "fixed_y_deterministic": fixed_y,
    }


In [ ]:
SIGMA_RATIO = 0.2
RUN_CONVERGENCE = True

uniform_metrics = evaluate_stochastic_value(
    duration_operation_year_scenarios, "uniform", SIGMA_RATIO,
    rp_solution=initial_solution
)

np.random.seed(123)
normal_durations = generate_scenarios(
    mean_duration_operation_year, N_scenarios,
    distribution="truncated_normal", sigma_ratio=SIGMA_RATIO
)
normal_metrics = evaluate_stochastic_value(
    normal_durations, "truncated_normal", SIGMA_RATIO
)

metrics = {
    "uniform": uniform_metrics,
    "truncated_normal": normal_metrics,
}

solutions = {
    "deterministic solution (uniform)": uniform_metrics["deterministic_solution"],
    "deterministic solution (truncated normal)": normal_metrics["deterministic_solution"],
    "RP (uniform)": uniform_metrics["rp_solution"],
    "RP (truncated normal)": normal_metrics["rp_solution"],
}

df_metrics = pd.DataFrame([
    {"distribution": name, **{k: values[k] for k in ("RP", "EEV", "VSS")}}
    for name, values in metrics.items()
]).set_index("distribution")
print(df_metrics.round(4).to_string())

df_freq = extract_frequency_data(uniform_metrics["rp_solution"])
df_costs = extract_cost_data(solutions)
target_freq = 1 / takt[0]

if RUN_CONVERGENCE:
    df_convergence = extract_convergence_data(
        ns=(5, 10, 15, 20), n_replications=1, seed=42
    )


In [ ]:
g = sns.catplot(
    data=df_freq,
    x="year",
    y="frequency",
    col="group",
    col_wrap=2,
    kind="box",
    sharey=False,
    height=4,
    aspect=1.2,
    color="lightblue"
)

g.map(
    plt.axhline,
    y=target_freq,
    color="#c44e52",
    linestyle="--",
    linewidth=2,
    zorder=5
)

g.set_axis_labels("years", "frequency (products/s)")
g.set_titles("group: {col_name}")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle(
    "stochastic variability of production frequency by resource group"
)

plt.show()

In [ ]:

plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_costs,
    x="approach",
    y="value",
    hue="cost_type",
    palette=["#4c72b0", "#dd8452"]
)

total_costs = (
    df_costs.groupby("approach", as_index=False)["value"].sum()
)

sns.scatterplot(
    data=total_costs,
    x="approach",
    y="value",
    color="black",
    marker="D",
    s=100,
    label="total cost",
    zorder=10
)

plt.title(
    "first-stage and expected second-stage costs"
)
plt.ylabel("cost (monetary unit)")
plt.xlabel("optimization model")
plt.legend(title="objective function component")
plt.tight_layout()
plt.show()

In [ ]:
if RUN_CONVERGENCE:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=df_convergence, x="n_scenarios", y="objective_value",
        hue="distribution", marker="o", linewidth=2,
        palette=["#55a868", "#c44e52"]
    )
    for distribution, color in (("uniform", "#2e7d32"), ("truncated_normal", "#9a303a")):
        plt.axhline(
            metrics[distribution]["deterministic_cost"], color=color, linestyle="--",
            linewidth=1.5, label=f"deterministic solution ({distribution.replace('_', ' ')})"
        )
    plt.title("in-sample SAA objective vs number of scenarios")
    plt.xlabel("number of scenarios (n)")
    plt.ylabel("optimal objective function value")
    plt.xscale("log")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# RP, EEV and VSS
labels = ["uniform", "truncated_normal"]
x = np.arange(len(labels))
width = 0.32
rp_values = [metrics[name]["RP"] for name in labels]
eev_values = [metrics[name]["EEV"] for name in labels]

fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(x - width / 2, rp_values, width, label="RP", color="#4c72b0")
ax.bar(x + width / 2, eev_values, width, label="EEV", color="#dd8452")
for i, name in enumerate(labels):
    top = max(rp_values[i], eev_values[i])
    ax.annotate(
        f"VSS = {metrics[name]['VSS']:,.2f}",
        xy=(x[i], top), xytext=(0, 8), textcoords="offset points",
        ha="center", va="bottom", fontsize=10
    )
ax.set_xticks(x, ["uniform", "truncated normal"])
ax.set_ylabel("expected cost (monetary unit)")
ax.set_title("value of the stochastic solution (VSS = EEV - RP)")
ax.legend()
ax.margins(y=0.17)
fig.tight_layout()
plt.show()
